In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/antimicrobial-resistance-prediction-from-maldi-tof/sample_submission.csv
/kaggle/input/antimicrobial-resistance-prediction-from-maldi-tof/species_mapping.csv
/kaggle/input/antimicrobial-resistance-prediction-from-maldi-tof/train.csv
/kaggle/input/antimicrobial-resistance-prediction-from-maldi-tof/test.csv


In [2]:
# ============================================================
# SCRIPT 3: CATBOOST MODEL (MODO CPU - MÁS SEGURO)
# ============================================================
import os
# Instalamos catboost silenciosamente
os.system('pip install -q catboost')

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.preprocessing import Normalizer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import warnings

warnings.filterwarnings('ignore')

# 1. CONFIGURACIÓN
ID_COL = "sample_id"
SPECIES_COL = "species_id"
ANTIBIOTICS = [
    "Ampicillin", "Levofloxacin", "Ciprofloxacin", "Imipenem",
    "Amoxicillin_Clavulanic_acid", "Ertapenem", "Cefotaxime", "Cefuroxime"
]

# --- CAMBIO PRINCIPAL: CONFIGURACIÓN PARA CPU ---
CAT_PARAMS = {
    'iterations': 1000,        # Un poco menos para que no tarde tanto en CPU
    'learning_rate': 0.05,     # Un poco más alto para compensar menos iteraciones
    'depth': 6,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    # 'task_type': 'GPU',      <-- BORRADO (Causaba el error)
    # 'devices': '0',          <-- BORRADO
    'random_seed': 42,
    'verbose': 200,
    'early_stopping_rounds': 50,
    'allow_writing_files': False,
    'thread_count': -1         # Usa todos los núcleos del procesador
}

# 2. CARGA DE DATOS
def load_data_catboost():
    print(">>> Cargando datos para CatBoost...")
    train = pd.read_csv("/kaggle/input/antimicrobial-resistance-prediction-from-maldi-tof/train.csv")
    test  = pd.read_csv("/kaggle/input/antimicrobial-resistance-prediction-from-maldi-tof/test.csv")
    
    spectrum_cols = [c for c in train.columns if c not in [ID_COL, SPECIES_COL] + ANTIBIOTICS]
    
    train['is_train'] = 1
    test['is_train'] = 0
    df_all = pd.concat([train, test], ignore_index=True)
    
    # Mantenemos las especies como TEXTO (Categórico)
    print(">>> Procesando Especies (Modo Categórico)...")
    df_all[SPECIES_COL] = df_all[SPECIES_COL].fillna("Unknown_Species").astype(str)
    
    print(">>> Normalizando Espectros...")
    spectrum_data = df_all[spectrum_cols].fillna(0)
    normalizer = Normalizer()
    spectrum_norm = pd.DataFrame(
        normalizer.fit_transform(spectrum_data),
        columns=spectrum_cols,
        index=df_all.index
    )
    
    X_all = pd.concat([spectrum_norm, df_all[[SPECIES_COL]]], axis=1)
    
    X_train = X_all[df_all['is_train'] == 1]
    X_test  = X_all[df_all['is_train'] == 0]
    y_train = train[ANTIBIOTICS]
    
    return X_train, y_train, X_test, test[ID_COL]

X, y, X_test, test_ids = load_data_catboost()

# 3. ENTRENAMIENTO
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
test_preds = np.zeros((X_test.shape[0], len(ANTIBIOTICS)))
scores = []

print("\n>>> Iniciando entrenamiento CatBoost (CPU)...")
CAT_FEATURES = [SPECIES_COL]

for i, antibiotic in enumerate(ANTIBIOTICS):
    print(f"\n=== Entrenando: {antibiotic} ===")
    
    valid_mask = y[antibiotic].notna()
    X_ab = X.loc[valid_mask]
    y_ab = y.loc[valid_mask, antibiotic].astype(int)
    
    fold_preds = []
    
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_ab, y_ab)):
        X_tr, X_val = X_ab.iloc[tr_idx], X_ab.iloc[val_idx]
        y_tr, y_val = y_ab.iloc[tr_idx], y_ab.iloc[val_idx]
        
        train_pool = Pool(data=X_tr, label=y_tr, cat_features=CAT_FEATURES)
        val_pool = Pool(data=X_val, label=y_val, cat_features=CAT_FEATURES)
        test_pool = Pool(data=X_test, cat_features=CAT_FEATURES)
        
        model = CatBoostClassifier(**CAT_PARAMS)
        
        model.fit(
            train_pool,
            eval_set=val_pool,
            use_best_model=True
        )
        
        p_test = model.predict_proba(test_pool)[:, 1]
        test_preds[:, i] += p_test / 5
        
        p_val = model.predict_proba(val_pool)[:, 1]
        score = roc_auc_score(y_val, p_val)
        fold_preds.append(score)
        
    print(f"     -> Avg AUC: {np.mean(fold_preds):.4f}")
    scores.append(np.mean(fold_preds))

print(f"\n>>> CatBoost Macro AUC Global: {np.mean(scores):.4f}")

# 4. SUBMISSION
sub_cat = pd.DataFrame(test_preds, columns=ANTIBIOTICS)
sub_cat.insert(0, ID_COL, test_ids.values)
sub_cat.to_csv("submission_cat.csv", index=False)
print("Archivo generado: submission_cat.csv")

>>> Cargando datos para CatBoost...
>>> Procesando Especies (Modo Categórico)...
>>> Normalizando Espectros...

>>> Iniciando entrenamiento CatBoost (CPU)...

=== Entrenando: Ampicillin ===
0:	test: 0.9090890	best: 0.9090890 (0)	total: 536ms	remaining: 8m 55s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9238784121
bestIteration = 108

Shrink model to first 109 iterations.
0:	test: 0.8908766	best: 0.8908766 (0)	total: 414ms	remaining: 6m 53s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9319943167
bestIteration = 66

Shrink model to first 67 iterations.
0:	test: 0.9241780	best: 0.9241780 (0)	total: 423ms	remaining: 7m 2s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9420242456
bestIteration = 146

Shrink model to first 147 iterations.
0:	test: 0.9113708	best: 0.9113708 (0)	total: 418ms	remaining: 6m 57s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9450542341
bestIteration = 118

Shrink model to fir